### ライブラリ

In [339]:
import pandas as pd
import numpy as np
import plotly.express as px
from scipy.optimize import least_squares
import plotly.graph_objects as go
from scipy.optimize import least_squares
from plotly.subplots import make_subplots
from scipy.optimize import brentq
from scipy.interpolate import CubicSpline
from scipy.interpolate import PchipInterpolator

### データ読み込み

In [340]:
# データの読み込み
datapath = fr"/Users/hisashi/python_projects/data_science/アプリ/data/fleetdata.xlsx"
df_fleetdata = pd.read_excel(datapath, sheet_name="DOD", index_col=0)
df_fleetdata

,0.010,0.250,0.500,0.750,0.900,0.950,0.990,0.998
TTP,,,,,,,,
0.010,10,20,25,30,40,45,50,51
0.250,12,21,26,31,41,45,51,52
0.500,14,22,26,32,43,45,52,53
0.750,16,24,28,33,42,45,53,54
0.900,18,25,27,33,43,45,54,56
0.950,20,27,29,35,44,45,53,57
0.990,22,28,30,34,45,46,53,58
0.998,24,28,32,34,47,48,54,59


In [341]:
data_path_TTPpercentile = fr"/Users/hisashi/python_projects/data_science/アプリ/data/TTP_Percentile.xlsx"
df_throughput_quantile = pd.read_excel(data_path_TTPpercentile)
df_throughput_quantile

,Percentile,Throughput
0,0.010,42
1,0.250,402
2,0.500,605
3,0.750,720
4,0.900,900
5,0.950,950
6,0.990,990
7,0.998,998


In [342]:
# 縦持ちデータに変換
df_fleetdata_long = df_fleetdata.stack().reset_index()
df_fleetdata_long.columns = [
    "ThroughputPercentile",
    "Percentile",
    "value"
]
df_fleetdata_long

,ThroughputPercentile,Percentile,value
0,0.010,0.010,10
1,0.010,0.250,20
2,0.010,0.500,25
3,0.010,0.750,30
4,0.010,0.900,40
...,...,...,...
59,0.998,0.750,34
60,0.998,0.900,47
61,0.998,0.950,48
62,0.998,0.990,54


In [343]:
# 可視化
fig = px.line(
    df_fleetdata_long,
    x="value",
    y="Percentile",
    color="ThroughputPercentile",
    markers=True,
)
fig.update_layout(
    width=600,
    height=400
)

### 確率モデルフィット（混合ワイブルstep1）

① DODと累積確率を取り出す
② 混合ワイブルCDFを定義する
③ eta, beta, weightの初期値を作る
④ least_squaresで実測CDFに近づくように推定する
⑤ 推定されたeta, beta, weightを返す

In [344]:
# =========================
# Weibull CDF
# =========================

def weibull_cdf(x, eta, beta):
    return 1 - np.exp(-(x / eta) ** beta)


def softmax(z):
    z = np.asarray(z)
    z = z - np.max(z)
    exp_z = np.exp(z)
    return exp_z / exp_z.sum()


# =========================
# K成分混合ワイブルCDF
# =========================

def mixture_weibull_cdf(x, params, n_components):
    """
    params:
        [log_eta_1, log_beta_1, ..., log_eta_K, log_beta_K, z_1, ..., z_K]
    """

    x = np.asarray(x, dtype=float)

    log_eta = params[0 : 2 * n_components : 2]
    log_beta = params[1 : 2 * n_components : 2]
    z_weight = params[2 * n_components : 3 * n_components]

    eta = np.exp(log_eta)
    beta = np.exp(log_beta)
    weights = softmax(z_weight)

    F = np.zeros_like(x, dtype=float)

    for k in range(n_components):
        F += weights[k] * weibull_cdf(x, eta[k], beta[k])

    return F


def residuals(params, x, p_obs, n_components):
    p_pred = mixture_weibull_cdf(x, params, n_components)
    return p_obs - p_pred


# =========================
# 初期値作成
# =========================

def make_initial_params(x, n_components):
    """
    etaはxの分位点でばらす。
    betaはすべて2.0。
    weightは均等。
    """

    quantiles = np.linspace(20, 80, n_components)
    eta_init = np.percentile(x, quantiles)
    beta_init = np.full(n_components, 2.0)

    log_eta_init = np.log(eta_init)
    log_beta_init = np.log(beta_init)

    z_weight_init = np.zeros(n_components)

    params = []

    for k in range(n_components):
        params.append(log_eta_init[k])
        params.append(log_beta_init[k])

    params.extend(z_weight_init)

    return np.array(params, dtype=float)


# =========================
# bounds作成
# =========================

def make_bounds(x, n_components):
    lower = []
    upper = []

    for _ in range(n_components):
        # eta
        lower.append(np.log(x.min() * 0.1))
        upper.append(np.log(x.max() * 10))

        # beta
        lower.append(np.log(0.1))
        upper.append(np.log(100))

    # mixture weight用のz
    for _ in range(n_components):
        lower.append(-20)
        upper.append(20)

    return np.array(lower), np.array(upper)


# =========================
# 1グループをK成分混合ワイブルでフィット
# =========================

def fit_mixture_weibull(df_group, n_components=2):
    df_group = df_group.sort_values("value").copy()

    x = df_group["value"].to_numpy(dtype=float)
    p_obs = df_group["Percentile"].to_numpy(dtype=float)

    # ワイブルなのでx > 0のみ使用
    mask = x > 0
    x = x[mask]
    p_obs = p_obs[mask]

    # 0,1は不安定なので丸める
    p_obs = np.clip(p_obs, 1e-6, 1 - 1e-6)

    if len(x) < 2 * n_components + n_components:
        return None

    init_params = make_initial_params(x, n_components)
    bounds = make_bounds(x, n_components)

    result = least_squares(
        residuals,
        x0=init_params,
        args=(x, p_obs, n_components),
        bounds=bounds,
        max_nfev=20000
    )

    params_hat = result.x

    log_eta = params_hat[0 : 2 * n_components : 2]
    log_beta = params_hat[1 : 2 * n_components : 2]
    z_weight = params_hat[2 * n_components : 3 * n_components]

    eta = np.exp(log_eta)
    beta = np.exp(log_beta)
    weights = softmax(z_weight)

    rmse = np.sqrt(np.mean(result.fun ** 2))

    output = {
        "n_components": n_components,
        "rmse": rmse,
        "success": result.success,
        "params": params_hat
    }

    for k in range(n_components):
        output[f"eta{k+1}"] = eta[k]
        output[f"beta{k+1}"] = beta[k]
        output[f"w{k+1}"] = weights[k]

    return output

# =========================
# ThroughputPercentileごとにフィット
# =========================

def fit_by_throughput_percentile(df_fleetdata_long, n_components=2):
    fit_results = []

    for tp, df_group in df_fleetdata_long.groupby("ThroughputPercentile"):

        fit = fit_mixture_weibull(
            df_group,
            n_components=n_components
        )

        if fit is None:
            continue

        fit["ThroughputPercentile"] = tp
        fit_results.append(fit)

    return pd.DataFrame(fit_results)


# =========================
# 可視化
# =========================

def plot_mixture_weibull_fit_compare(
    df_fleetdata_long,
    df_fit_results_dict,
    n_components_list=(1, 2, 3),
    width=1800,
    height=600
):
    fig = make_subplots(
        rows=1,
        cols=len(n_components_list),
        subplot_titles=[
            f"{n} component Weibull" for n in n_components_list
        ],
        shared_yaxes=False
    )

    for col_idx, n_components in enumerate(n_components_list, start=1):

        df_fit_results = df_fit_results_dict.get(n_components)

        # フィット結果が丸ごとない場合はスキップ
        if df_fit_results is None or df_fit_results.empty:
            fig.add_annotation(
                text=f"No fit result<br>{n_components} component",
                x=0.5,
                y=0.5,
                showarrow=False,
                row=1,
                col=col_idx
            )
            continue

        for tp, df_group in df_fleetdata_long.groupby("ThroughputPercentile"):

            fit_row = df_fit_results[
                df_fit_results["ThroughputPercentile"] == tp
            ]

            # そのThroughputPercentileのフィット結果がない場合はスキップ
            if fit_row.empty:
                continue

            params = fit_row.iloc[0].get("params", None)

            # paramsがない、または欠損の場合はスキップ
            if params is None:
                continue

            try:
                params = np.asarray(params, dtype=float)
            except Exception:
                continue

            if params.size != 3 * n_components:
                continue

            df_group = df_group.sort_values("value").copy()

            x = df_group["value"].to_numpy(dtype=float)
            p_obs = df_group["Percentile"].to_numpy(dtype=float)

            mask = x > 0
            x = x[mask]
            p_obs = p_obs[mask]

            if len(x) < 2:
                continue

            x_grid = np.linspace(x.min(), x.max(), 300)

            try:
                p_fit = mixture_weibull_cdf(
                    x_grid,
                    params,
                    n_components=n_components
                )
            except Exception:
                continue

            # 実測点
            fig.add_trace(
                go.Scatter(
                    x=x,
                    y=p_obs,
                    mode="markers",
                    name=f"Obs TP={tp}",
                    legendgroup=f"TP={tp}",
                    # showlegend=(col_idx == 1),
                    showlegend=False,
                    marker=dict(size=6)
                ),
                row=1,
                col=col_idx
            )

            # フィット線
            fig.add_trace(
                go.Scatter(
                    x=x_grid,
                    y=p_fit,
                    mode="lines",
                    name=f"Fit TP={tp}",
                    legendgroup=f"TP={tp}",
                    showlegend=False
                ),
                row=1,
                col=col_idx
            )

        fig.update_xaxes(
            title_text="DOD",
            row=1,
            col=col_idx
        )

        fig.update_yaxes(
            title_text="Cumulative Probability",
            row=1,
            col=col_idx
        )

    fig.update_layout(
        width=width,
        height=height,
        legend_title="Throughput Percentile"
    )

    return fig


# =========================
# QQプロット
# =========================

def mixture_weibull_ppf(p, params, n_components, x_min=1e-12, x_max=None):
    """
    混合ワイブルCDFの逆関数。
    F(x) = p となる x を数値的に解く。
    """

    p = float(np.clip(p, 1e-10, 1 - 1e-10))

    if x_max is None:
        log_eta = params[0 : 2 * n_components : 2]
        eta = np.exp(log_eta)
        x_max = np.max(eta) * 100

    def func(x):
        return mixture_weibull_cdf(
            np.array([x]),
            params,
            n_components
        )[0] - p

    # 上限が足りない場合に広げる
    while func(x_max) < 0:
        x_max *= 2

    return brentq(func, x_min, x_max)


def plot_mixture_weibull_qq_compare(
    df_fleetdata_long,
    df_fit_results_dict,
    n_components_list=(1, 2, 3),
    width=1800,
    height=600
):
    fig = make_subplots(
        rows=1,
        cols=len(n_components_list),
        subplot_titles=[
            f"{n} component Weibull QQ" for n in n_components_list
        ],
        shared_yaxes=False
    )

    for col_idx, n_components in enumerate(n_components_list, start=1):

        df_fit_results = df_fit_results_dict.get(n_components)

        if df_fit_results is None or df_fit_results.empty:
            fig.add_annotation(
                text=f"No fit result<br>{n_components} component",
                x=0.5,
                y=0.5,
                showarrow=False,
                row=1,
                col=col_idx
            )
            continue

        qq_x_all = []
        qq_y_all = []

        for tp, df_group in df_fleetdata_long.groupby("ThroughputPercentile"):

            fit_row = df_fit_results[
                df_fit_results["ThroughputPercentile"] == tp
            ]

            if fit_row.empty:
                continue

            params = fit_row.iloc[0].get("params", None)

            if params is None:
                continue

            try:
                params = np.asarray(params, dtype=float)
            except Exception:
                continue

            if params.size != 3 * n_components:
                continue

            df_group = df_group.sort_values("value").copy()

            x_obs = df_group["value"].to_numpy(dtype=float)
            p_obs = df_group["Percentile"].to_numpy(dtype=float)

            mask = (x_obs > 0) & (p_obs > 0) & (p_obs < 1)
            x_obs = x_obs[mask]
            p_obs = p_obs[mask]

            if len(x_obs) < 2:
                continue

            x_theory = []

            for p in p_obs:
                try:
                    q = mixture_weibull_ppf(
                        p=p,
                        params=params,
                        n_components=n_components,
                        x_max=max(x_obs.max() * 10, 1.0)
                    )
                    x_theory.append(q)
                except Exception:
                    x_theory.append(np.nan)

            x_theory = np.asarray(x_theory, dtype=float)

            valid = np.isfinite(x_theory) & np.isfinite(x_obs)

            if valid.sum() < 2:
                continue

            x_theory = x_theory[valid]
            x_obs_valid = x_obs[valid]

            qq_x_all.extend(x_theory)
            qq_y_all.extend(x_obs_valid)

            fig.add_trace(
                go.Scatter(
                    x=x_theory,
                    y=x_obs_valid,
                    mode="markers",
                    name=f"TP={tp}",
                    legendgroup=f"TP={tp}",
                    # showlegend=(col_idx == 1)
                    showlegend=False
                ),
                row=1,
                col=col_idx
            )

        # 45度線
        if len(qq_x_all) > 0 and len(qq_y_all) > 0:
            min_val = min(min(qq_x_all), min(qq_y_all))
            max_val = max(max(qq_x_all), max(qq_y_all))

            fig.add_trace(
                go.Scatter(
                    x=[min_val, max_val],
                    y=[min_val, max_val],
                    mode="lines",
                    name="45 degree line",
                    showlegend=False,
                    # showlegend=(col_idx == 1),
                    line=dict(dash="dash")
                ),
                row=1,
                col=col_idx
            )

        fig.update_xaxes(
            title_text="Theoretical quantile",
            row=1,
            col=col_idx
        )

        fig.update_yaxes(
            title_text="Observed DOD",
            row=1,
            col=col_idx
        )

    fig.update_layout(
        width=width,
        height=height,
        # title="QQ plot",
        legend_title="Throughput Percentile"
    )

    return fig

### 精度確認

In [345]:
df_fit_1 = fit_by_throughput_percentile(
    df_fleetdata_long,
    n_components=1
)

df_fit_2 = fit_by_throughput_percentile(
    df_fleetdata_long,
    n_components=2
)

df_fit_3 = fit_by_throughput_percentile(
    df_fleetdata_long,
    n_components=3
)

df_fit_results_dict = {
    1: df_fit_1,
    2: df_fit_2,
    3: df_fit_3
}

fig = plot_mixture_weibull_fit_compare(
    df_fleetdata_long,
    df_fit_results_dict,
    n_components_list=(1, 2, 3)
)

fig.show()

In [346]:
df_fit_results_dict = {
    1: df_fit_1,
    2: df_fit_2,
    3: df_fit_3
}

fig_qq = plot_mixture_weibull_qq_compare(
    df_fleetdata_long,
    df_fit_results_dict,
    n_components_list=(1, 2, 3)
)

fig_qq.show()

In [347]:
# 精度比較

def safe_mean_rmse(df):
    if df is None:
        return np.nan
    
    if not isinstance(df, pd.DataFrame):
        return np.nan
    
    if df.empty:
        return np.nan
    
    if "rmse" not in df.columns:
        return np.nan
    
    return df["rmse"].mean()


summary = pd.DataFrame({
    "n_components": [1, 2, 3],
    "mean_rmse": [
        safe_mean_rmse(df_fit_1),
        safe_mean_rmse(df_fit_2),
        safe_mean_rmse(df_fit_3),
    ]
})

summary

,n_components,mean_rmse
0,1,0.044548
1,2,0.007663
2,3,NaN


In [348]:
df_fit_results_dict[2]

,n_components,rmse,success,params,eta1,beta1,w1,eta2,beta2,w2,ThroughputPercentile
0,2,0.003675,True,"[3.255815116002667, 1.4603750804328721, 3.8229...",25.940751,4.307575,0.881182,45.737237,11.683052,0.118818,0.010
1,2,0.005983,True,"[3.2814963796573737, 1.5331680254701354, 3.806...",26.615570,4.632831,0.854522,45.003893,9.469896,0.145478,0.250
2,2,0.005035,True,"[3.2466437337194276, 1.7989601684159486, 3.772...",25.703926,6.043360,0.762812,43.486284,12.875390,0.237188,0.500
3,2,0.005150,True,"[3.315764534511498, 1.888219177417368, 3.72218...",27.543444,6.607591,0.709892,41.354638,6.082762,0.290108,0.750
4,2,0.004140,True,"[3.287059117586322, 2.563122289091691, 3.76501...",26.764038,12.976270,0.739976,43.164422,12.018622,0.260024,0.900
5,2,0.004071,True,"[3.360509691428318, 2.6313644950940995, 3.7875...",28.803868,13.892714,0.749325,44.148967,25.016694,0.250675,0.950
6,2,0.003614,True,"[3.39483379138582, 2.668886820678709, 3.810201...",29.809698,14.423904,0.750802,45.159518,25.698647,0.249198,0.990
7,2,0.029639,True,"[3.4759720376858634, 2.246105243587124, 3.9356...",32.329239,9.450855,0.905426,51.194477,18.614571,0.094574,0.998


### 確率モデルフィット（混合ワイブルstep2）

In [349]:
def build_global_mixture_weibull_cdf(
    df_fit_results_dict,
    n_components=2,
    q_col="ThroughputPercentile"
):
    """
    df_fit_results_dict から、任意の ThroughputPercentile q に対する
    混合ワイブルCDF関数を作成する。

    Parameters
    ----------
    df_fit_results_dict : dict
        {1: df_fit_1, 2: df_fit_2, 3: df_fit_3} のような辞書

    n_components : int
        使用する混合ワイブルの成分数

    q_col : str
        ThroughputPercentile列名

    Returns
    -------
    global_mixture_weibull_cdf : function
        global_mixture_weibull_cdf(x, q)
    """

    df_fit = df_fit_results_dict.get(n_components)

    if df_fit is None or df_fit.empty:
        raise ValueError(f"{n_components}成分のフィット結果がありません。")

    df_fit = df_fit.sort_values(q_col).copy()

    q_values = df_fit[q_col].to_numpy(dtype=float)

    eta_splines = []
    beta_splines = []
    weight_splines = []

    for k in range(1, n_components + 1):
        eta_col = f"eta{k}"
        beta_col = f"beta{k}"
        w_col = f"w{k}"

        if eta_col not in df_fit.columns:
            raise ValueError(f"{eta_col} が df_fit にありません。")
        if beta_col not in df_fit.columns:
            raise ValueError(f"{beta_col} が df_fit にありません。")
        if w_col not in df_fit.columns:
            raise ValueError(f"{w_col} が df_fit にありません。")

        eta_values = df_fit[eta_col].to_numpy(dtype=float)
        beta_values = df_fit[beta_col].to_numpy(dtype=float)
        w_values = df_fit[w_col].to_numpy(dtype=float)

        # 正値制約のため log で補間
        eta_splines.append(
            CubicSpline(q_values, np.log(eta_values), extrapolate=True)
        )

        beta_splines.append(
            CubicSpline(q_values, np.log(beta_values), extrapolate=True)
        )

        # weightは後で正規化する前提でそのまま補間
        weight_splines.append(
            CubicSpline(q_values, w_values, extrapolate=True)
        )

    def global_mixture_weibull_cdf(x, q):
        x = np.asarray(x, dtype=float)

        eta = np.array([
            np.exp(spline(q)) for spline in eta_splines
        ])

        beta = np.array([
            np.exp(spline(q)) for spline in beta_splines
        ])

        weights = np.array([
            spline(q) for spline in weight_splines
        ], dtype=float)

        # weightが負になったり合計1からズレるのを防ぐ
        weights = np.clip(weights, 0, None)

        if weights.sum() == 0:
            weights = np.ones(n_components) / n_components
        else:
            weights = weights / weights.sum()

        F = np.zeros_like(x, dtype=float)

        for k in range(n_components):
            F += weights[k] * weibull_cdf(
                x,
                eta[k],
                beta[k]
            )

        return F

    return global_mixture_weibull_cdf

In [350]:
model_weibull = build_global_mixture_weibull_cdf(
    df_fit_results_dict,
    n_components=2
)

p_pred = model_weibull(
    x = 50,   # DOD
    q = 0.10  # ThroughputPercentile
)

p_pred  # ThroughputPercentile=10%の時にDOD50%以下となる確率

array(0.98777287)

### 乱数生成

In [351]:
def build_bivariate_sampler(
    df_throughput_quantile,
    model,
    percentile_col="Percentile",
    throughput_col="Throughput"
):
    """
    Throughput分布と DOD|ThroughputPercentile 分布から、
    (Throughput, DOD) の二変数乱数生成器を作る。
    """

    df_q = df_throughput_quantile[[percentile_col, throughput_col]].copy()
    df_q = df_q.dropna().sort_values(percentile_col)

    q_values = df_q[percentile_col].to_numpy(dtype=float)
    t_values = df_q[throughput_col].to_numpy(dtype=float)

    # 1,25,50... の表記なら 0.01,0.25,0.50... に変換
    if q_values.max() > 1:
        q_prob = q_values / 100
    else:
        q_prob = q_values.copy()

    # Throughputの分位点関数：q -> throughput
    throughput_ppf = PchipInterpolator(
        q_prob,
        t_values,
        extrapolate=True
    )

    # ThroughputのCDF：throughput -> q
    throughput_cdf = PchipInterpolator(
        t_values,
        q_prob,
        extrapolate=True
    )

    def sample_dod_given_q(q, rng, x_min=1e-12, x_max_init=None):
        """
        DOD | q から1点サンプリング
        """
        u = rng.uniform(1e-10, 1 - 1e-10)

        if q_values.max() > 1:
            q_for_global = q * 100
        else:
            q_for_global = q

        if x_max_init is None:
            x_max = 1.0
        else:
            x_max = x_max_init

        def func(x):
            return float(model(x, q_for_global)) - u

        # 上限が足りなければ広げる
        while func(x_max) < 0:
            x_max *= 2

            if x_max > 1e12:
                raise RuntimeError("DODの探索上限が大きくなりすぎました。")

        return brentq(func, x_min, x_max)

    def sample(n_samples=1000, random_state=None):
        rng = np.random.default_rng(random_state)

        # Step1: Throughputの累積確率 q を一様乱数で生成
        q_random = rng.uniform(
            q_prob.min(),
            q_prob.max(),
            size=n_samples
        )

        # Step2: q -> Throughput に変換
        throughput_samples = throughput_ppf(q_random)

        # Step3: Throughput -> q に戻す
        # 数値誤差や補間の一貫性確認のため。基本的には q_random と近い。
        q_for_dod = throughput_cdf(throughput_samples)
        q_for_dod = np.clip(q_for_dod, q_prob.min(), q_prob.max())

        # Step4: DOD | q から生成
        dod_samples = []

        # 探索上限の初期値
        x_max_init = max(1.0, np.nanmax(throughput_samples))

        for q in q_for_dod:
            dod = sample_dod_given_q(
                q=q,
                rng=rng,
                x_max_init=x_max_init
            )
            dod_samples.append(dod)

        df_sample = pd.DataFrame({
            "ThroughputPercentile": q_for_dod * 100,
            "Throughput": throughput_samples,
            "DOD": dod_samples
        })

        return df_sample

    return sample

In [352]:
sampler = build_bivariate_sampler(
    df_throughput_quantile=df_throughput_quantile,
    model=model_weibull,
    percentile_col="Percentile",
    throughput_col="Throughput"
)

df_sample = sampler(
    n_samples=10000,
    random_state=42
)

df_sample

,ThroughputPercentile,Throughput,DOD
0,77.432678,741.256486,31.312754
1,44.487620,569.213416,29.445102
2,85.895607,848.415604,23.520413
3,69.706022,690.138452,16.278951
4,9.078160,203.138575,21.676049
...,...,...,...
9995,45.265020,574.470253,38.480954
9996,31.421427,466.852757,26.106638
9997,16.969120,308.431049,35.279013
9998,37.894190,521.297393,28.353588


### 精度確認

In [353]:
def global_cdf_ppf(global_cdf, p, q, x_min=1e-12, x_max_init=1.0):
    """
    global_cdf(x, q) = p となる x を数値的に解く
    """
    p = float(np.clip(p, 1e-10, 1 - 1e-10))
    upper = x_max_init

    def func(x):
        return float(global_cdf(x, q)) - p

    while func(upper) < 0:
        upper *= 2
        if upper > 1e12:
            raise RuntimeError("探索上限が大きくなりすぎました。")

    return brentq(func, x_min, upper)


def plot_generated_sample_diagnostics(
    df_sample,
    global_cdf,
    throughput_col="Throughput",
    dod_col="DOD",
    q_col="ThroughputPercentile",
    nbins=50,
    width=1800,
    height=600
):
    """
    生成サンプルの診断プロットを横一列に表示する。

    1. 散布図：Throughput vs DOD
    2. ヒストグラム：DOD分布
    3. Q-Qプロット：サンプルDOD vs 理論分位点
    """

    df = df_sample[[throughput_col, dod_col, q_col]].dropna().copy()

    throughput = df[throughput_col].to_numpy(dtype=float)
    dod = df[dod_col].to_numpy(dtype=float)
    q_values = df[q_col].to_numpy(dtype=float)

    # =========================
    # Q-Qプロット用データ作成
    # =========================
    df_qq = df.sort_values(dod_col).copy()

    dod_sorted = df_qq[dod_col].to_numpy(dtype=float)
    q_sorted = df_qq[q_col].to_numpy(dtype=float)

    n = len(dod_sorted)

    # 経験分位点
    p_emp = (np.arange(1, n + 1) - 0.5) / n

    theoretical_quantiles = []

    x_max_init = max(1.0, np.nanmax(dod_sorted) * 2)

    for p, q in zip(p_emp, q_sorted):
        try:
            x_theory = global_cdf_ppf(
                global_cdf=global_cdf,
                p=p,
                q=q,
                x_max_init=x_max_init
            )
        except Exception:
            x_theory = np.nan

        theoretical_quantiles.append(x_theory)

    theoretical_quantiles = np.asarray(theoretical_quantiles, dtype=float)

    valid = (
        np.isfinite(theoretical_quantiles)
        & np.isfinite(dod_sorted)
    )

    theoretical_quantiles = theoretical_quantiles[valid]
    dod_qq = dod_sorted[valid]

    # Q-Q用の45度線
    if len(theoretical_quantiles) > 0:
        qq_max = max(
            np.nanmax(theoretical_quantiles),
            np.nanmax(dod_qq)
        )
    else:
        qq_max = np.nanmax(dod)

    qq_max = qq_max * 1.05

    # =========================
    # subplot作成
    # =========================
    fig = make_subplots(
        rows=1,
        cols=3,
        subplot_titles=[
            "Scatter plot",
            "DOD histogram",
            "Q-Q plot"
        ]
    )

    # =========================
    # 1. 散布図
    # =========================
    fig.add_trace(
        go.Scatter(
            x=throughput,
            y=dod,
            mode="markers",
            name="Sample",
            marker=dict(size=5, opacity=0.4),
            showlegend=False
        ),
        row=1,
        col=1
    )

    # =========================
    # 2. ヒストグラム
    # =========================
    fig.add_trace(
        go.Histogram(
            x=dod,
            nbinsx=nbins,
            name="DOD",
            opacity=0.75,
            showlegend=False
        ),
        row=1,
        col=2
    )

    # =========================
    # 3. Q-Qプロット
    # =========================
    fig.add_trace(
        go.Scatter(
            x=theoretical_quantiles,
            y=dod_qq,
            mode="markers",
            name="Q-Q",
            marker=dict(size=5, opacity=0.5),
            showlegend=False
        ),
        row=1,
        col=3
    )

    fig.add_trace(
        go.Scatter(
            x=[0, qq_max],
            y=[0, qq_max],
            mode="lines",
            name="45 degree line",
            line=dict(dash="dash"),
            showlegend=False
        ),
        row=1,
        col=3
    )

    # =========================
    # 軸設定
    # =========================

    fig.update_xaxes(
        title_text="Throughput",
        range=[0, np.nanmax(throughput) * 1.05],
        row=1,
        col=1
    )
    fig.update_yaxes(
        title_text="DOD",
        range=[0, np.nanmax(dod) * 1.05],
        row=1,
        col=1
    )

    fig.update_xaxes(
        title_text="DOD",
        range=[0, np.nanmax(dod) * 1.05],
        row=1,
        col=2
    )
    fig.update_yaxes(
        title_text="Count",
        row=1,
        col=2
    )

    fig.update_xaxes(
        title_text="Theoretical DOD",
        range=[0, qq_max],
        row=1,
        col=3
    )
    fig.update_yaxes(
        title_text="Sample DOD",
        range=[0, qq_max],
        row=1,
        col=3,
        scaleanchor="x3",
        scaleratio=1
    )

    fig.update_layout(
        width=width,
        height=height,
        # title="Generated sample diagnostics",
        bargap=0.05
    )

    return fig

In [354]:
fig = plot_generated_sample_diagnostics(
    df_sample=df_sample,
    global_cdf=model_weibull,
    throughput_col="Throughput",
    dod_col="DOD",
    q_col="ThroughputPercentile",
    nbins=50
)

fig.show()

/var/folders/mr/f3kxn5992mg1223gwtf66dcc0000gn/T/ipykernel_23153/4207517032.py:74: RuntimeWarning: overflow encountered in exp
  np.exp(spline(q)) for spline in eta_splines


### 分位点補正

In [355]:
def global_cdf_ppf(global_cdf, p, q, x_min=1e-12, x_max_init=1.0):
    """
    global_cdf(x, q) = p となる x を数値的に解く
    """
    p = float(np.clip(p, 1e-10, 1 - 1e-10))
    upper = x_max_init

    def func(x):
        return float(global_cdf(x, q)) - p

    while func(upper) < 0:
        upper *= 2
        if upper > 1e12:
            raise RuntimeError("探索上限が大きくなりすぎました。")

    return brentq(func, x_min, upper)


def quantile_correction(
    df_sample,
    global_cdf,
    dod_col="DOD",
    q_col="ThroughputPercentile",
    throughput_col="Throughput"
):
    """
    分位点補正を行う関数。

    入力:
        df_sample:
            Throughput, ThroughputPercentile, DOD を持つDataFrame

        global_cdf:
            global_cdf(x, q) -> F(DOD <= x | q)

    出力:
        df_corrected:
            元データに以下を追加
            - EmpiricalPercentile
            - TheoreticalDOD
            - CorrectedDOD
            - CorrectionAmount
    """

    df = df_sample.copy()
    df = df.dropna(subset=[dod_col, q_col]).copy()

    # DODを昇順に並べる
    df_sorted = df.sort_values(dod_col).copy()

    n = len(df_sorted)

    # 経験分位点
    p_emp = (np.arange(1, n + 1) - 0.5) / n

    dod_sorted = df_sorted[dod_col].to_numpy(dtype=float)
    q_sorted = df_sorted[q_col].to_numpy(dtype=float)

    x_max_init = max(1.0, np.nanmax(dod_sorted) * 2)

    theoretical_dod = []

    for p, q in zip(p_emp, q_sorted):
        try:
            x_theory = global_cdf_ppf(
                global_cdf=global_cdf,
                p=p,
                q=q,
                x_max_init=x_max_init
            )
        except Exception:
            x_theory = np.nan

        theoretical_dod.append(x_theory)

    theoretical_dod = np.asarray(theoretical_dod, dtype=float)

    df_sorted["EmpiricalPercentile"] = p_emp
    df_sorted["TheoreticalDOD"] = theoretical_dod

    # 分位点補正後のDOD
    df_sorted["CorrectedDOD"] = df_sorted["TheoreticalDOD"]

    # 補正量
    df_sorted["CorrectionAmount"] = (
        df_sorted["CorrectedDOD"] - df_sorted[dod_col]
    )

    return df_sorted

In [356]:
df_corrected = quantile_correction(
    df_sample=df_sample,
    global_cdf=model_weibull,
    dod_col="DOD",
    q_col="ThroughputPercentile",
    throughput_col="Throughput"
)

df_corrected.head()

/var/folders/mr/f3kxn5992mg1223gwtf66dcc0000gn/T/ipykernel_23153/4207517032.py:74: RuntimeWarning: overflow encountered in exp
  np.exp(spline(q)) for spline in eta_splines


,ThroughputPercentile,Throughput,DOD,EmpiricalPercentile,TheoreticalDOD,CorrectedDOD,CorrectionAmount
217,12.313472,248.891843,1.650597,0.00005,NaN,NaN,NaN
194,15.623065,291.779593,4.202241,0.00015,NaN,NaN,NaN
9327,5.569414,146.214871,4.494603,0.00025,NaN,NaN,NaN
7349,13.020556,258.328561,5.303127,0.00035,NaN,NaN,NaN
4411,4.905620,134.077848,5.330731,0.00045,NaN,NaN,NaN
